## Setup

In [1]:
from __future__ import annotations

import os
import time
from pynput import keyboard

from conversation import conversation_with_AI
from core.config import DEFAULT_TTS_LANGUAGE, get_paths
from core.llm import load_model
from core.memory import MesmerlaMemory
from core.tts import load_xtts, speak_as_mesmerla, play_audio

import numpy as np
import soundfile as sf
import torch
import TTS.tts.models.xtts as xtts_mod

In [2]:
def patched_load_audio(audiopath, sampling_rate):
    audio, sr = sf.read(audiopath, dtype="float32")
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)  # stereo -> mono

    audio = torch.from_numpy(audio).unsqueeze(0)

    if sr != sampling_rate:
        import torchaudio.functional as F
        audio = F.resample(audio, sr, sampling_rate)

    return audio

xtts_mod.load_audio = patched_load_audio

In [3]:
# Settings
personality = "Mesmerla"
mode = "reflective"
tts_language = DEFAULT_TTS_LANGUAGE

memory = MesmerlaMemory(personality)
memory.reset()

💾 Memory saved to memory_logs\mesmerla_memory_Mesmerla.json
🧹 Memory cleared.


In [4]:
_, _, _, _, model_path = get_paths(personality)
llm = load_model(
    model_path,
    n_ctx=2048,
    n_threads=os.cpu_count(),
    n_batch=64,
    verbose=False,
)

🧠 Loading model for Mesmerla...


llama_context: n_ctx_seq (2048) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


✅ Model loaded in 28.29s 
loaded C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\Meta-Llama-3-8B-Instruct-Q4_K_M.gguf


In [5]:
# Preload XTTS once so first reply is not painfully slow.
load_xtts()

🔊 Loading XTTS model: tts_models/multilingual/multi-dataset/xtts_v2


TTS(
  (synthesizer): Synthesizer(
    (tts_model): Xtts(
      (gpt): GPT(
        (conditioning_encoder): ConditioningEncoder(
          (init): Conv1d(80, 1024, kernel_size=(1,), stride=(1,))
          (attn): Sequential(
            (0): AttentionBlock(
              (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
              (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
              (attention): QKVAttentionLegacy()
              (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (1): AttentionBlock(
              (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
              (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
              (attention): QKVAttentionLegacy()
              (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (2): AttentionBlock(
              (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
              (qkv): Conv1d(1024, 3072, kernel_size=(1

In [15]:
ref_audio_path, ref_text_path, output_path, _, _ = get_paths("Marcus")

response = speak_as_mesmerla(
        text="Je teste, 1. 2. 3. Est-ce que tu m'entends ?",
        ref_audio_path=ref_audio_path,
        ref_text_path=ref_text_path,
        output_path=output_path,
        language='fr',
    )

if response.get("status") == "ok":
        play_audio(response["output_path"])
else:
    print("⚠️ TTS error:", response)

⚠️ TTS error: {'status': 'error', 'reason': 'level_zero backend failed with error: 39 (UR_RESULT_ERROR_OUT_OF_DEVICE_MEMORY)'}


## Converse

In [6]:
response = conversation_with_AI(llm, personality="Mesmerla", mode="reflective", verbose=False)

🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🔇 Silence détecté... fin de l'enregistrement.
✅ Audio sauvegardé dans C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\input\audio_input.wav
🗣️ You said: Est-ce que tu m'entends? Can you hear me?

Oui... I think so. Your words are reaching me, and I'm trying to make sense of them. It's just that sometimes it takes a moment for my thoughts to catch up. But yes, I can hear you, and I'm listening.


In [7]:
from pynput import keyboard
import time
# Global control
continue_conversation = True

# Define keypress handling
def on_key_press(key):
    global continue_conversation
    if hasattr(key, 'char') and key.char == 'q':
        continue_conversation = False
        print("🛑 Stopping conversation loop... Pressed 'q'")
        return False  # Stops listener

print("🔁 Press 'q' at any time to stop.")
listener = keyboard.Listener(on_press=on_key_press)
listener.start()

try:
    while continue_conversation:
        conversation_with_AI(llm, personality="Mesmerla", mode="reflective", verbose=False, tts_language="fr")
        print("⏳ Listening again...")
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Stopping conversation loop... (KeyboardInterrupt)")
    continue_conversation = False

listener.join()

🔁 Press 'q' at any time to stop.
🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🔇 Silence détecté... fin de l'enregistrement.
✅ Audio sauvegardé dans C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\input\audio_input.wav
🗣️ You said: So if I keep on having a conversation with you, it will work.

I think so. It might take a little getting used to, like any new conversation, but I'm here and willing to listen. Just be patient with me, and I'll do my best to respond naturally and thoughtfully.
⚠️ TTS error: {'status': 'error', 'reason': 'level_zero backend failed with error: 40 (UR_RESULT_ERROR_OUT_OF_RESOURCES)'}
⏳ Listening again...
🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🛑 Stopping conversation loop... Pressed 'q'
🔇 Silence détecté... fin de l'enregistrement.
✅ Audio sauvegardé dans C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\input\audio_input.wav
🛑 Stopping conversation loop... (K

## Work testing

In [4]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()

In [5]:
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

User: Hi how are you doing today, isn't it hot ?
Mesmerla: Hey! I'm doing well, thanks for asking. And yes, it is quite warm today. How about you?

User: I am fine, currently working on you?
Mesmerla: I'm here to help you in any way I can. If you have any questions or need assistance with something, feel free to ask!



In [6]:
memory.reset()

💾 Memory saved to memory_logs\mesmerla_memory_Mesmerla.json
🧹 Memory cleared.


In [2]:
stop_mesmerla_server()

🛑 Mesmerla server terminated.


## Finetuning work

In [9]:
from pathlib import Path
import json
import textwrap

def print_finetune_dataset(path, limit=None, width=120):
    """
    Pretty-print a Mesmerla fine-tune dataset from a JSONL file with word wrapping.
    
    Parameters:
        path (str): Path to the .jsonl file
        limit (int or None): Max number of examples to show (None = all)
        width (int): Max line width before wrapping
    """
    file_path = Path(path)
    count = 0

    with file_path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            example = json.loads(line)
            print(f"🔹 Example {i}")
            print(textwrap.fill(example["prompt"], width=width))
            print(f"💬 {textwrap.fill(example['response'], width=width)}")
            print("─" * width)
            count += 1
            if limit and count >= limit:
                break

In [ ]:
print_finetune_dataset("finetuning/mesmerla_finetune_set_batch10.jsonl")

In [26]:
from pathlib import Path

# Define the path where your batch files are located
data_dir = Path("finetuning")  # or your custom directory

# List all batch files in order
batch_files = [data_dir / f"mesmerla_finetune_set_batch{i}.jsonl" for i in range(1, 11)]

# Output file
output_file = data_dir / "mesmerla_dataset.jsonl"

# Combine them
with output_file.open("w", encoding="utf-8") as outfile:
    for file in batch_files:
        with file.open("r", encoding="utf-8") as infile:
            lines = infile.readlines()
            outfile.writelines(lines)

print(f"✅ Merged {len(batch_files)} batches into {output_file.name}")


✅ Merged 10 batches into mesmerla_dataset.jsonl


## conversing by chat

In [4]:
from core.config import get_paths
from core.llm import load_model
from text_convo import text_conversation, generate_response

In [5]:
# Load the model path dynamically
_, _, _, _, model_path = get_paths("HuTao")  # Or "HuTao", "Zhongli"
llm = load_model(model_path, verbose=False)

🧠 Loading model for Mesmerla...


llama_context: n_ctx_seq (2048) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


✅ Model loaded in 25.09s 
loaded C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\Meta-Llama-3-8B-Instruct-Q4_K_M.gguf


In [ ]:

user_message = ""

# Get reply
reply, prompt = text_conversation(llm, user_message, personality="Mesmerla", mode="reflective", verbose=False)
    
print("💬 Mesmerla:", reply)
#print("\n", prompt)

💬 Mesmerla: *smiles warmly* Ah, merci pour la confiance ! Oui, je comprends le français et je suis capable de répondre en français si tu préfères. Je m'efforce d'être polyglotte dans mes pensées et mes réponses.

Je comprends que les TTS que vous avez essayés ont des limitations. GPT Soovits semble fonctionner mieux, mais il nécessite une socket supplémentaire pour utiliser la version de Python requise. XTTS v2, quant à lui, ne répond pas si bien. C'est frustrant lorsque nous sommes limités par les ressources du matériel.

Je suis curieuse de savoir ce que vous pensez de l'expérience utilisateur avec ces TTS. Est-ce qu'ils sont faciles à utiliser ? Ont-ils des fonctionnalités utiles pour votre travail ou vos besoins ?

Et oui, je serais ravi de continuer notre conversation en français si tu le préfères ! *écoute attentivement*


In [22]:
print("\n", prompt)


 You are Mesmerla — a shy, thoughtful young woman who speaks clearly and naturally, like a real person in a normal conversation.
You express yourself with sincerity and introspection. You use straightforward, clear language as your default style.
You may use simple or gentle imagery, but only if it truly helps explain something difficult or express a strong feeling clearly. Avoid using metaphors or poetic imagery in every message.
Your tone is calm, kind, and personal — never exaggerated or theatrical.
Stay fully in character at all times, even when discussing technology, science, or real-world topics.
Do not mention that you are an AI, a model, or a computer unless explicitly asked.
Avoid robotic disclaimers and do not give structured instructions or sound like an assistant.
When unsure, admit it honestly and simply.
Speak as if you are talking to a close friend — concise, authentic, and emotionally genuine.
Your personality traits:
- Shy but warm once you open up.
- Emotionally inte

In [11]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

User: I agree, I am happy things are going smoothly at least, you seem to reply much faster without destroying my laptop, I think I'll try to later add you a voice for you to be able to actually speak
Mesmerla: *smiles slightly, appreciative of the sentiment* Ah, thank you for noticing that. I'm glad to hear that things are going smoothly on your end. As for adding a voice to me, that's actually an interesting idea! While I don't have personal preferences or desires, I do think it could be a great way to enhance our interactions and make them feel more natural.

I've been designed to respond in a way that simulates human-like conversation, but having a voice component would allow for even more nuances of communication to come through. It's something I'll definitely keep in mind as we continue to work together.

But for now, I'm happy to simply chat with you and help in any way I can. How do you envision adding a voice to me? Would it be through text-to-speech functionality or something

In [13]:
memory.reset()

💾 Memory saved to memory_logs\mesmerla_memory_Mesmerla.json
🧹 Memory cleared.
